# Current corrected IEEE-CIS evaluation

## 1. Objective
Compare a fixed Logistic Regression baseline with fixed XGBoost on later transactions. This notebook presents **aggregate outputs of the completed run in `results/current_evaluation.json`**. It does not retrain, open restricted raw rows, create a submission or select a better score.

The historical ROC-AUC `0.9421992865326172` belongs to the previous preprocessing workflow and is not evidence for this corrected temporal study. Earlier demonstration code remains available in Git history. No Kaggle leaderboard result is claimed.

## 2. Data contract
Transaction tables only; identity tables and sample submission were deliberately not used. The competition test table is unlabeled and supplies contract/provenance checks, not performance labels. Restricted CSVs remain outside Git. Counts and hashes below come from actual execution.

In [1]:
import json
from pathlib import Path
import pandas as pd
from evaluation import validate_result

result = json.loads(Path("results/current_evaluation.json").read_text())
validate_result(result)
assert result["status"] == "completed" and result["dataset_kind"] == "ieee"
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

def table(rows):
    print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.6f}"))

summary = result["dataset_summary"]
print("Status:", result["publication_status"])
print("Rows:", result["row_counts"])
print("Raw/transformed predictors:", summary["predictor_count"], summary["transformed_feature_count"])
print("Target / ID:", summary["target"], summary["id_field"])
print("File hashes:", result["provenance"]["data_sha256"])

Status: FULL EMPIRICAL RUN
Rows: {'train_csv': 590540, 'test_csv': 506691}
Raw/transformed predictors: 392 392
Target / ID: isFraud TransactionID
File hashes: {'train_transaction.csv': '3a5c83ab6b3cc13dcabe5ffa9f522307fd5f7f7b6e6f6a60c32284ca6283d642', 'test_transaction.csv': '2a8e51f1d335a86025d2b7f45beb9b78d0ab1edd726ef531d8b71a8a0065c011'}


## 3. Class imbalance
Fraud prevalence contextualizes ranking and alert precision. ROC-AUC does not by itself state the positive yield at a review budget. Average Precision (AP) summarizes precision-recall ranking; it is not every possible trapezoidal PR-area definition.

In [2]:
print("Frauds / nonfrauds:", summary["positive_count"], summary["negative_count"])
print(f"Full training prevalence: {summary['target_prevalence']:.6f}")
print(f"Final evaluation prevalence: {result['metrics']['xgboost']['prevalence']:.6f}")

Frauds / nonfrauds: 20663 569877
Full training prevalence: 0.034990
Final evaluation prevalence: 0.034409


## 4. Validation design
Primary: earliest approximately 60% model-training, middle 20% calibration, latest 20% final evaluation. Equal timestamps stay together and boundaries are strictly ordered. Both classes are present in each partition. Stratified feasibility was checked but no alternate model score was selected. Card attributes are not verified entity IDs, so no invented group split was used.

The runner reads the full training table and checks labels/class feasibility before its metric-decision lock. This is not a physically sealed-label holdout. Competition test transactions occur later still, but no test labels are available.

In [3]:
table([{"partition": name, **{k: p[k] for k in ("rows", "positive_count", "time_min", "time_max")}} for name, p in result["split"]["partitions"].items()])
print("Competition test strictly later:", result["split"]["test_strictly_after_train"])

  partition   rows  positive_count        time_min        time_max
model_train 354324           11988    86400.000000  8745772.000000
calibration 118108            4611  8745798.000000 12192842.000000
 evaluation 118108            4064 12192900.000000 15811131.000000
Competition test strictly later: True


## 5. Preprocessing and leakage boundaries
Target and TransactionID are excluded. Numeric medians, categorical imputers/vocabularies and baseline scaling are fitted only on model-training rows. Unseen categories map to -1. The existing dtype-derived numerical treatment of card codes is retained. Ordinal nominal distances constrain the linear baseline; this is not an optimized one-hot Logistic Regression benchmark.

Mechanical safeguards do not prove decision-time availability for anonymized C/D/M/V features. Exact target/ID-copy and suspicious-name checks passed; duplicate and proxy recurrence evidence belongs to the supplemental leakage review. No blanket absence-of-leakage claim is made.

In [4]:
print("Preprocessing fit:", result["preprocessing"]["fit_partition"])
print("Baseline scaler fit:", result["preprocessing"]["baseline_scaler_fit_partition"])
print("Exact repeated predictor rows:", result["leakage_audit"]["duplicate_predictor_rows"])
print("Known forbidden/suspicious:", result["leakage_audit"]["known_forbidden_features"], result["leakage_audit"]["suspicious_feature_names"])
print("Feature timing:", result["leakage_audit"]["feature_timing"])

Preprocessing fit: model_train only
Baseline scaler fit: model_train only
Exact repeated predictor rows: 3
Known forbidden/suspicious: [] []
Feature timing: unknown for anonymized variables; availability at prediction time is not verified


### Actual related-observation audit
Complete card tuples are unverified attribute proxies, not entity IDs. Duplicate checks do not redesign the split or remove rows. The following aggregates identify dependence limits; no blanket no-leakage claim is made.


In [5]:
audit = json.loads(Path("results/leakage_review.json").read_text())
assert audit["input_sha256"] == result["provenance"]["data_sha256"]
proxy = audit["card_tuple_proxy"]["partitions"]["evaluation"]
print("Complete evaluation card tuples matching train:", proxy["rows_matching_train_complete_tuple"], "/", proxy["complete_tuple_rows"])
print("Fraction of complete evaluation rows matching training proxy:", f"{proxy['fraction_complete_rows_matching_train']:.6f}")
for variant, checks in audit["duplicate_checks"].items():
    if isinstance(checks, dict):
        print(variant, "duplicates:", checks["duplicate_rows_excluding_first"], "cross-partition groups:", checks["cross_partition_duplicate_groups"], "evaluation rows matching training:", checks["evaluation_rows_exactly_matching_train_predictors"])
print("Feature timing and label maturity remain uncertain; no verified entity-disjoint guarantee.")

Complete evaluation card tuples matching train: 113953 / 115926
Fraction of complete evaluation rows matching training proxy: 0.982981
with_TransactionDT duplicates: 3 cross-partition groups: 0 evaluation rows matching training: 0
without_TransactionDT duplicates: 1503 cross-partition groups: 463 evaluation rows matching training: 192
Feature timing and label maturity remain uncertain; no verified entity-disjoint guarantee.


## 6. Baseline
Fixed, unweighted, scaled Logistic Regression (C=1, max_iter=1000, seed 42). The convergence gate passed without warnings. This model has poor probability quality and particularly poor top-budget precision on this future cohort; it is not silently tuned or replaced.

In [6]:
metric_names = ("roc_auc", "average_precision", "precision", "recall", "f1", "brier", "log_loss")
table([{ "model": "Logistic Regression", **{k: result["metrics"]["logistic_regression"][k] for k in metric_names}}])

              model  roc_auc  average_precision  precision   recall       f1    brier  log_loss
Logistic Regression 0.805437           0.173031   0.325390 0.262057 0.290309 0.040453  0.497000


## 7. XGBoost
Fixed 500-tree/depth-6/learning-rate-0.05 histogram XGBoost; row/column subsampling 0.8, seed 42, CPU threads 4. No hyperparameter search, identity enrichment, early stopping on final outcomes or score-recovery tuning was performed.

In [7]:
table([{ "model": "XGBoost", **{k: result["metrics"]["xgboost"][k] for k in metric_names}}])

  model  roc_auc  average_precision  precision   recall       f1    brier  log_loss
XGBoost 0.887546           0.480197   0.770548 0.332185 0.464237 0.023365  0.097501


## 8. Evaluation and baseline comparison
Ranking metrics, threshold-0.5 metrics and probability-quality metrics answer different questions. Compare models on identical final rows. Confusion-matrix ordering is [[TN, FP], [FN, TP]]. Improvements describe this cohort, not guaranteed future performance.

In [8]:
table([{ "metric": k, "logistic_regression": result["metrics"]["logistic_regression"][k], "xgboost": result["metrics"]["xgboost"][k], "xgboost_minus_baseline": result["metrics"]["xgboost"][k] - result["metrics"]["logistic_regression"][k]} for k in metric_names])
for name, metrics in result["metrics"].items():
    print(name, "confusion matrix at 0.5:", metrics["confusion_matrix"])

           metric  logistic_regression  xgboost  xgboost_minus_baseline
          roc_auc             0.805437 0.887546                0.082108
average_precision             0.173031 0.480197                0.307167
        precision             0.325390 0.770548                0.445158
           recall             0.262057 0.332185                0.070128
               f1             0.290309 0.464237                0.173927
            brier             0.040453 0.023365               -0.017089
         log_loss             0.497000 0.097501               -0.399499
logistic_regression confusion matrix at 0.5: [[111836, 2208], [2999, 1065]]
xgboost confusion matrix at 0.5: [[113642, 402], [2714, 1350]]


## 9. Investigation capacity
Top 0.5%, 1% and 2% were fixed before final outcomes. Alerts are ceiling-rounded; equal scores use original-row-index tie breaking. These are batch ranking scenarios, not staffing promises or guaranteed future cutoff capacity. No monetary loss/prevention claims are made.

The baseline's tightest-budget cutoff is saturated at probability 1.0 and precision is below cohort prevalence. This documents extreme-score unreliability; it does not identify a unique cause or justify retuning after final evaluation.

In [9]:
table([{ "model": name, "review_rate": b["budget_fraction"], "alerts": b["alerts"], "frauds_captured": b["tp"], "false_positives": b["fp"], "precision": b["precision"], "recall": b["recall"], "score_cutoff": b["threshold"]} for name, metrics in result["metrics"].items() for b in metrics["review_budgets"]])

              model  review_rate  alerts  frauds_captured  false_positives  precision   recall  score_cutoff
logistic_regression     0.005000     591                4              587   0.006768 0.000984      1.000000
logistic_regression     0.010000    1182               34             1148   0.028765 0.008366      1.000000
logistic_regression     0.020000    2363              724             1639   0.306390 0.178150      0.806987
            xgboost     0.005000     591              548               43   0.927242 0.134843      0.965198
            xgboost     0.010000    1182             1029              153   0.870558 0.253199      0.771379
            xgboost     0.020000    2363             1551              812   0.656369 0.381644      0.339076


## 10. Probability calibration
The predeclared trigger was calibration-partition ECE > 0.02 with at least 1,000 rows and 30 cases per class. The measured calibration ECE did not exceed the trigger, so sigmoid was **not** fitted. Raw and selected XGBoost are therefore identical. ECE is bin-dependent; a low overall value and lower Brier do not certify high-risk or production calibration.

In [10]:
calibration = result["calibration"]
print("Pre-final calibration ECE:", f"{calibration['before']['reliability']['ece']:.6f}")
print("Decision:", calibration["decision"])
table([{ "model": name, "brier": m["brier"], "log_loss": m["log_loss"], "ece": m["reliability"]["ece"]} for name, m in result["metrics"].items()])
for name, m in result["metrics"].items():
    print(name, "final reliability bins")
    table(m["reliability"]["bins"])

Pre-final calibration ECE: 0.005008
Decision: no calibration: exploratory flag not exceeded; adequacy not established
              model    brier  log_loss      ece
logistic_regression 0.040453  0.497000 0.035347
            xgboost 0.023365  0.097501 0.004580
logistic_regression final reliability bins
 count  positives  probability_mean  fraud_fraction
 11811        122          0.002858        0.010329
 11811         82          0.006935        0.006943
 11811         95          0.010628        0.008043
 11810        118          0.015928        0.009992
 11811        147          0.022885        0.012446
 11811        190          0.030923        0.016087
 11810        244          0.041549        0.020660
 11811        365          0.059270        0.030903
 11811        553          0.091773        0.046821
 11811       2148          0.399844        0.181864
xgboost final reliability bins
 count  positives  probability_mean  fraud_fraction
 11811         28          0.001937     

## 11. Feature interpretation
Normalized total gain measures predictive contribution, not causation. Leading V258 and C-family fields remain timing-uncertain; their masked names do not establish underlying meaning. TransactionDT has predictive contribution and represents time/cohort information, not a transaction identifier. No feature was removed or optimized based on final scores.

In [11]:
table(result["importance"]["features"][:10])

                feature   total_gain  normalized_total_gain
          numeric__V258 57232.367188               0.101292
           numeric__C14 26848.437500               0.047517
            numeric__C1 26713.738281               0.047279
           numeric__C13 22874.117188               0.040484
 numeric__TransactionDT 20048.791016               0.035483
         numeric__card1 18346.695312               0.032471
          numeric__V294 14344.192383               0.025387
numeric__TransactionAmt 13229.150391               0.023413
            numeric__C8 12044.750977               0.021317
         numeric__card2 11944.455078               0.021140


## 12. Limitations
- Anonymized feature availability and fraud-label maturation are unresolved; no arbitrary embargo was invented.
- Repeated card-attribute proxies can cross boundaries; this is not unseen-customer validation.
- One chronological cohort, no confidence interval, no robustness score search.
- Ordinal/dtype-based representation limits the Logistic Regression comparator.
- Baseline extreme probabilities perform poorly; XGBoost also overpredicts the final highest decile on average.
- Calibration-to-final performance degradation indicates cohort sensitivity, not guaranteed deployable drift handling.
- No identity-table enrichment, leaderboard evidence, deployment, operational intervention or monetary-savings validation.
- Original full-data integrity is supported by readable dimensions and matching archive/file hashes, not independently signed official checksums.
- No production-readiness claim.

## 13. Reproducibility
The executable study is `evaluation.py`; this notebook replays aggregate evidence only. Authorized users should keep raw CSVs outside Git, use the pinned environment, and run the documented command into a **new** result filename because completed evidence is protected.

```bash
uv sync --locked --no-dev --extra notebook
uv run --no-sync python evaluation.py --data-dir /path/to/authorized/ieee --output /path/to/new_evaluation.json --split temporal --n-jobs 4 --bootstrap-replicates 0 --acknowledge-uncertain-feature-timing
uv run --no-sync python scripts/validate_results.py /path/to/new_evaluation.json
```

CI replays small/synthetic tests and validates evidence; it never trains on restricted competition data.

In [12]:
print("Execution Git SHA:", result["provenance"]["git_sha"])
print("Clean at execution:", result["provenance"]["worktree_clean"])
print("Created UTC:", result["created_at_utc"])
print("Python / libraries:", result["provenance"]["python"], result["provenance"]["versions"])
print("Runtime seconds:", f"{result['runtime_seconds']:.3f}")
print("Warnings:", result["warnings"])

Execution Git SHA: 7148defa53109f6e47cc6993d6fbc4bd55d9c3ef
Clean at execution: True
Created UTC: 2026-10-02T22:51:03.575182+00:00
Python / libraries: 3.11.16 {'numpy': '2.4.6', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'xgboost': '3.2.0'}
Runtime seconds: 204.817
Warnings: []
